In [24]:
import pandas as pd
import numpy as np

import requests
import os
from dotenv import load_dotenv

import concurrent.futures
import threading
import time

In [25]:
#########################################
# STEP 1: CLEAN CSV TO UPLOAD NEW DATA  #
#########################################


df2 = pd.read_csv('unique_songs.csv')
df2['spotify track id'] = df2['track spotify url'].str.split(':', n=2).str[-1]

# create new columns for api call variables
added_columns = [
    "api track id", "id", "key", "mode", "camelot", "tempo", "duration",
    "popularity", "energy", "danceability", "happiness",
    "acousticness", "instrumentalness", "liveness",
    "speechiness", "loudness"
]

for col in added_columns:
    df2[col] = None

df2

,track title,track artist,track spotify url,spotify track id,api track id,id,key,mode,camelot,tempo,duration,popularity,energy,danceability,happiness,acousticness,instrumentalness,liveness,speechiness,loudness
0,Tear in My Heart,Twenty One Pilots,spotify:track:3bnVBN67NBEzedqQuWrpP4,3bnVBN67NBEzedqQuWrpP4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,Fun (feat. Tove Lo),Coldplay,spotify:track:7fJFDK6XjYsXcMKNHESbot,7fJFDK6XjYsXcMKNHESbot,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,Someone Like You,Adele,spotify:track:4kflIGfjdZJW4ot2ioixTB,4kflIGfjdZJW4ot2ioixTB,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,Immortals,Fall Out Boy,spotify:track:3Te8uLyit6X3ncNW8Fp3K2,3Te8uLyit6X3ncNW8Fp3K2,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,Scary Monsters and Nice Sprites,Skrillex,spotify:track:4rwpZEcnalkuhPyGkEdhu0,4rwpZEcnalkuhPyGkEdhu0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27429,in the sea,kensuke ushio,spotify:track:6hShfxTdYcC7YQ3AhfRedv,6hShfxTdYcC7YQ3AhfRedv,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27430,ジェーンは教会で眠った,レゼ（上田麗奈）,spotify:track:1YS1h55ne33IOw3u1CG2UD,1YS1h55ne33IOw3u1CG2UD,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27431,JANE DOE,Kenshi Yonezu,spotify:track:4oE7MyJhqSD3BaHRpNs8Nl,4oE7MyJhqSD3BaHRpNs8Nl,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
27432,Garden Waltz,Alstad,spotify:track:4wuQUHu2WQu3jSkK9wCRFY,4wuQUHu2WQu3jSkK9wCRFY,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [26]:
#########################################
# STEP 2: SET UP FOR GET API FUNCTION   #
#########################################
load_dotenv()
api_key = os.getenv("RAPIDAPI_KEY")

# initialize lock
lock = threading.Lock()
        
# set up api call session
headers = {
        'x-rapidapi-key': api_key,
        'x-rapidapi-host': 'track-analysis.p.rapidapi.com'
    }
session = requests.Session()
session.headers.update(headers)

# keep track of failed track requests
failed_tracks = []

In [27]:
#########################################
# STEP 3: DEFINE GET API FUNCTION       #
#########################################

# 0 = under limit, 1 = limit hit
api_limit = 0

def get_api_data(i):
    # set spotifyId based on index i
    spotifyId = df2.at[i,'spotify track id']
    
    # configure api request
    url = f"https://track-analysis.p.rapidapi.com/pktx/spotify/{spotifyId}"
    
    try:
        # make api call
        result = session.get(url, timeout=40)
        
        # error check
        if result.status_code == 200:   # okay call
            output = result.json()
        elif result.status_code == 429: # 
            output = {}
            api_limit = 1
        else:
            output = {}
            print(f"Error {result.status_code}: \n{result.text}")
            with lock:
                failed_tracks.append(i)
    except requests.exceptions.RequestException as ex:
        output = {}
        print(f"API Call failed for track {i}: {ex}")
        with lock:
            failed_tracks.append(i)
    
    
    # thread safety to avoid corrupt data
    with lock:
        df2.at[i,'api track id'] = spotifyId
        for col in added_columns:
            if col == "api track id":
                df2.at[i,col] = spotifyId
            else:
                df2.at[i,col] = output.get(col)
    
    # buffer for rate limit
    time.sleep(3)
            
    # debugging setup
    return i

In [31]:
#########################################
# STEP 4a: SET UP BATCH VARIABLES       #
#########################################

# batch variables
batch_start = 0
batch_size = 1
total_tracks = 27434

In [29]:
#########################################
# STEP 4b: SAVE HEADER IN CSV FILE      #
#########################################

df2.iloc[0:0].to_csv('progress_backup.csv', mode='w', header=True, index=False )

In [ ]:
#########################################
# STEP 4b: ITERATE API CALLS IN BATCHES #
#########################################

# check range of batch & assign batch_end value
if (batch_end + batch_size) > total_tracks:
    batch_end = total_tracks
else:
batch_end = batch_start + batch_size

# run get_api_data for all tracks, 1 batch (100 api calls) at a time
print(f"SONGS: {batch_start} THROUGH {batch_end}")

for task in range(batch_start, batch_end):    
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(get_api_data, i) for i in range(batch_start,batch_end)]
        
        for count, task in enumerate(concurrent.futures.as_completed(futures)):
            task.result()
    
    # save this batch's rows into csv file
    df2.iloc[batch_start:batch_end].to_csv('progress_backup.csv', mode='a', header=False, index=False )
    
    

    #     batch_end = batch_start + batch_size

# signal end of api calls
print("Finished processing all tracks!")
print(f"{len(failed_tracks)} failed.")
print(failed_tracks)

# for next run 
batch_start += batch_size



SONGS: 0 THROUGH 1
Finished processing all tracks!
0 failed.
[]


In [ ]:
#########################################
# STEP 5: FILL ROWS WITH API ERRORS     #
#########################################

# df3 = pd.read_csv('progress_backup3.csv')

# for i in range(1,df3.shape[0]):
#     # check to see which rows of sheet do not include api data
#     if df3.iloc[i,4] == None:
#         # run api call again
#         print(f"Retry: Row {i}")
#         get_api_data(i)  

# df3 